


# 9. Keras and deep learning
In this lab, we will learn how to use Keras to build deep learning models. We will use Keras to build a LSTM model for sentiment classification and a CNN model for digit recognition.
You need to put in the code to complete the models in the blocks marked with `## YOUR CODE HERE` and `## END OF YOUR CODE`.

## Installation
Before you can start using Keras, you'll need to install TensorFolw, which includes Keras as part of its core library.
```bash
source activate {your_env}
pip install tensorflow
pip install keras
```

## Basics of Keras
Keras is a high-level neural networks API, written in Python and capable of running on top of TensorFlow, CNTK, or Theano. It was developed with a focus on enabling fast experimentation. Being able to go from idea to result with the least possible delay is key to doing good research.

### 1. Initialize a model
Start by creating a Sequential model and adding layers to it.
```python
from keras.models import Sequential
from keras.layers import Dense

# Initialize a model
model = Sequential()

# Add layers to the model
model.add(Dense(units=64, activation='relu', input_dim=100))
model.add(Dense(units=10, activation='softmax'))

# this is equivalent to the above
#model = Sequential([
#    Dense(64, activation='relu', input_dim=100),
#    Dense(10, activation='softmax')
#])
```


In [22]:
from keras.models import Sequential
from keras.layers import Dense
import numpy as np

# Initialize a model
model = Sequential()

# Add layers to the model
model.add(Dense(units=64, activation='relu', input_dim=100))
model.add(Dense(units=10, activation='softmax'))

### 2. Compile the model
Compile the model with the appropriate loss function and optimizer.
```python
model.compile(loss='categorical_crossentropy', # loss function, binary_crossentropy for binary classification
              optimizer='sgd', # stochastic gradient descent
              metrics=['accuracy'])
```


In [23]:
model.compile(loss='categorical_crossentropy', # loss function, binary_crossentropy for binary classification
              optimizer='sgd', # stochastic gradient descent
              metrics=['accuracy'])

### 3. Train the model
Train the model with the training data.
```python
x_train = np.random.random((1000, 100))
y_train = np.random.randint(2, size=(1000, 10))
model.fit(x_train, y_train, epochs=5, batch_size=32)
```


In [24]:
x_train = np.random.random((1000, 100))
y_train = np.random.randint(2, size=(1000, 10))
model.fit(x_train, y_train, epochs=5, batch_size=32)

Epoch 1/5
32/32 [==============================] - 0s 2ms/step - loss: 12.5271 - accuracy: 0.0680
Epoch 2/5
32/32 [==============================] - 0s 2ms/step - loss: 60.6546 - accuracy: 0.0800
Epoch 3/5
32/32 [==============================] - 0s 2ms/step - loss: 3931.8452 - accuracy: 0.1340
Epoch 4/5
32/32 [==============================] - 0s 2ms/step - loss: 163564.9688 - accuracy: 0.1150
Epoch 5/5
32/32 [==============================] - 0s 2ms/step - loss: 8729349.0000 - accuracy: 0.0860


### 4. Evaluate the model
Evaluate the model with the test data.
```python
x_test = np.random.random((100, 100))
y_test = np.random.randint(2, size=(100, 10))
loss_and_metrics = model.evaluate(x_test, y_test, batch_size=128)
```


In [25]:
x_test = np.random.random((100, 100))
y_test = np.random.randint(2, size=(100, 10))
loss_and_metrics = model.evaluate(x_test, y_test, batch_size=128)

1/1 [==============================] - 0s 151ms/step - loss: 64892820.0000 - accuracy: 0.2000


## Keras LSTM for IMDB sentiment classification
The IMDB dataset is in `datasets/` of this repository. Use the following code the load the dataset and write a LSTM model to classify the sentiment of the reviews.
```python
import pandas as pd    # to load dataset
import nltk
from nltk.corpus import stopwords   # to get a collection of stopwords

data = pd.read_csv('../datasets/IMDB.csv')

custom_path = '../datasets/'

# Append your custom path to the NLTK data path
nltk.data.path.append(custom_path)

nltk.download('stopwords', download_dir=custom_path)
english_stops = set(stopwords.words('english'))

x_data = data['review']       # Reviews/Input
y_data = data['sentiment']    # Sentiment/Output
# PRE-PROCESS REVIEW
x_data = x_data.replace({'<.*?>': ''}, regex = True)          # remove html tag
x_data = x_data.replace({'[^A-Za-z]': ' '}, regex = True)     # remove non alphabet
x_data = x_data.apply(lambda review: [w for w in review.split() if w not in english_stops])  # remove stop words
x_data = x_data.apply(lambda review: [w.lower() for w in review])   # lower case
```


In [26]:
import pandas as pd    # to load dataset
import nltk
from nltk.corpus import stopwords   # to get a collection of stopwords
from google.colab import drive
import pandas as pd

# Mount Google Drive
#drive.mount('/content/drive')

data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/TopicsInMathematics/Labs/Lab9/IMDB.csv")

custom_path = '/content/drive/MyDrive/Colab Notebooks/TopicsInMathematics/Labs/Lab9/'

# Append your custom path to the NLTK data path
nltk.data.path.append(custom_path)

nltk.download('stopwords', download_dir=custom_path)
english_stops = set(stopwords.words('english'))

x_data = data['review']       # Reviews/Input
y_data = data['sentiment']    # Sentiment/Output
# PRE-PROCESS REVIEW
x_data = x_data.replace({'<.*?>': ''}, regex = True)          # remove html tag
x_data = x_data.replace({'[^A-Za-z]': ' '}, regex = True)     # remove non alphabet
x_data = x_data.apply(lambda review: [w for w in review.split() if w not in english_stops])  # remove stop words
x_data = x_data.apply(lambda review: [w.lower() for w in review])   # lower case



[nltk_data] Downloading package stopwords to
[nltk_data]     /content/drive/MyDrive/Colab
[nltk_data]     Notebooks/TopicsInMathematics/Labs/Lab9/...
[nltk_data]   Package stopwords is already up-to-date!


The tokenization of the reviews is done by the following code:
```python
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
tokenizer = Tokenizer(num_words=10000)    # num_words is the number of words to keep based on word frequency
tokenizer.fit_on_texts(x_data)            # fit tokenizer to our training text data

# retrieve the word index
word_index = tokenizer.word_index

x_data = tokenizer.texts_to_sequences(x_data)  # convert our text data to sequence of numbers
```


In [27]:
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
tokenizer = Tokenizer(num_words=10000)    # num_words is the number of words to keep based on word frequency
tokenizer.fit_on_texts(x_data)            # fit tokenizer to our training text data

# retrieve the word index
word_index = tokenizer.word_index

x_data = tokenizer.texts_to_sequences(x_data)  # convert our text data to sequence of numbers

Now, complete the following code to create a LSTM model for the IMDB sentiment classification.

In [18]:
import numpy as np
from keras.models import Sequential
from keras.layers import Embedding, SimpleRNN, LSTM, Dense, GRU
from sklearn.model_selection import train_test_split
# from keras.utils import np_utils

# Pad sequences to ensure uniform input size
max_length = 100  # You can choose a different length
x_data = pad_sequences(x_data, maxlen=max_length)

# Convert sentiments to numerical labels
y_data = np.where(y_data == 'positive', 1, 0)

# Split data into training and test sets
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.2, random_state=42)

## YOUR CODE HERE
# Build the RNN model
model = Sequential()
model.add(Embedding(input_dim = 10000, output_dim = 64, input_length = max_length))
model.add(SimpleRNN(32))
model.add(Dense(1, activation='sigmoid'))
# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.2)  # You can adjust epochs and batch_size

## END OF YOUR CODE

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {test_acc}")

Epoch 1/5
250/250 [==============================] - 11s 39ms/step - loss: 0.0359 - accuracy: 0.9972 - val_loss: 0.0028 - val_accuracy: 1.0000
Epoch 2/5
250/250 [==============================] - 12s 49ms/step - loss: 0.0016 - accuracy: 1.0000 - val_loss: 9.7343e-04 - val_accuracy: 1.0000
Epoch 3/5
250/250 [==============================] - 12s 49ms/step - loss: 7.0853e-04 - accuracy: 1.0000 - val_loss: 5.2054e-04 - val_accuracy: 1.0000
Epoch 4/5
250/250 [==============================] - 11s 44ms/step - loss: 4.1435e-04 - accuracy: 1.0000 - val_loss: 3.3129e-04 - val_accuracy: 1.0000
Epoch 5/5
313/313 [==============================] - 2s 7ms/step - loss: 2.3095e-04 - accuracy: 1.0000
Test Accuracy: 1.0


In [11]:
from google.colab import drive
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Directory path
directory = '/content/drive/MyDrive/Colab Notebooks/TopicsInMathematics/Labs/Digits/'


# Read the CSV files into DataFrames
X_train = pd.read_csv(directory + 'Digits_X_train.csv').values
y_train = pd.read_csv(directory + 'Digits_y_train.csv').values
X_test  = pd.read_csv(directory + 'Digits_X_test.csv').values
y_test  = pd.read_csv(directory + 'Digits_y_test.csv').values

Mounted at /content/drive


Complete the following code to create a CNN model for the digit recognition.

In [38]:
from keras.models import Sequential
from keras.layers import Dense, Conv2D, Flatten, MaxPooling2D
from tensorflow import keras

# Reshape the data to 8 * 8 * 1
X_train = X_train.reshape(X_train.shape[0], 8, 8, 1)
X_test = X_test.reshape(X_test.shape[0], 8, 8, 1)
# Reshape y_train to match the batch size
y_train = keras.utils.to_categorical(y_train, num_classes=10)  # Assuming 10 classes

## YOUR CODE HERE
# Create the model
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(8, 8, 1)))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(10, activation='softmax'))
# Print the model summary
print(model.summary())

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=5, batch_size=32, validation_split=0.2)

## END OF YOUR CODE

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print("Accuracy: ", accuracy)

Model: "sequential_13"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_16 (Conv2D)          (None, 6, 6, 32)          320       
                                                                 
 max_pooling2d_9 (MaxPoolin  (None, 3, 3, 32)          0         
 g2D)                                                            
                                                                 
 conv2d_17 (Conv2D)          (None, 1, 1, 64)          18496     
                                                                 
 flatten_7 (Flatten)         (None, 64)                0         
                                                                 
 dense_21 (Dense)            (None, 128)               8320      
                                                                 
 dense_22 (Dense)            (None, 10)                1290      
                                                     

ValueError: in user code:

    File "/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py", line 1401, in train_function  *
        return step_function(self, iterator)
    File "/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py", line 1384, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py", line 1373, in run_step  **
        outputs = model.train_step(data)
    File "/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py", line 1151, in train_step
        loss = self.compute_loss(x, y, y_pred, sample_weight)
    File "/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py", line 1209, in compute_loss
        return self.compiled_loss(
    File "/usr/local/lib/python3.10/dist-packages/keras/src/engine/compile_utils.py", line 277, in __call__
        loss_value = loss_obj(y_t, y_p, sample_weight=sw)
    

(1347, 8, 8, 1)
(1000, 10)
(450, 8, 8, 1)
(100, 10)


In [48]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D

# Assuming num_classes is defined somewhere
num_classes = 10

# Define the input dimensions
img_rows, img_cols = 28, 28
input_shape = (img_rows, img_cols, 1)

# Reshape the data and normalize
x_train = x_train.reshape(-1, img_rows, img_cols, 1)
x_test = x_test.reshape(-1, img_rows, img_cols, 1)
x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255

# Define the model
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3),
                 activation='relu',
                 input_shape=input_shape))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

# Compile the model
model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# Define batch size and epochs
batch_size = 128
epochs = 10

# Train the model
model.fit(x_train, y_train,
          batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(x_test, y_test))

# Evaluate the model
score = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

# Save the model
model.save("test_model.h5")


ValueError: cannot reshape array of size 100000 into shape (28,28,1)

In [45]:
print(x_train.shape)
print(x_test.shape)


(1000, 100)
(100, 100)
